In [15]:
using Plots

In [3]:
# ================ #
#  SET PARAMETERS  #
# ================ #

alpha = 0.40;   # capital share
delta = 0.08;   # depreciation rate
beta = 0.98;    # discount factor
rho = 0.5;      # SS replacement rate
rho_ss1 = 0.25;      # replacement rate at the final steady state 

NT = 100; # transition period

# iteration parameters
maxiter = 2000; # max number of iterations
tol = 1e-3;     # error tolerance level
adjK = 0.2;     # adjustment factor of capital in updating guess

# grid construction
Nj = 61;        # max age: 61 years max (age 20-80)
Njw = 45;       # working years (retire at Njw+1: enter at 21 and retire at 65)
Na = 201;       # asset state
Na2 = 8001;     # asset choice
Ne = 2;         # labor productivity (low/high)

minA = 0.0;       # min asset grid
maxA = 25.0;      # max asset grid
curvA = 1.2;    # asset grid density (=1 eequal size)

# labor productivity grid (mean=1)
gride = zeros(Ne);
dif = 0.3;
gride[1] = 1.0 - dif;
gride[2] = 1.0 + dif;

# labor productivity transition matrix
Pe = zeros(Ne,Ne);
Pe[1,1] = 0.8;
Pe[1,2] = 1.0 - Pe[1,1];

Pe[2,2] = copy(Pe[1,1]);
Pe[2,1] = 1.0 - Pe[2,2];


# age distribution (assume no death and no pop growth. sum to 1)
meaJ = 1/Nj .* ones(Nj);


# aggregate labor supply
L = sum(meaJ[1:Njw]);

In [4]:
# create constructer that contains parameters
struct Model{TI<:Integer, TF<:AbstractFloat}
    
    alpha::TF   # capital share
    delta::TF   # depreciation rate
    beta::TF    # discount factor
    rho::TF      # SS replacement rate

    # iteration parameters
    maxiter::TI # max number of iterations
    tol::TF     # error tolerance level
    adjK::TF     # adjustment factor of capital in updating guess

    # grid construction
    Nj::TI        # max age: 61 years max (age 20-80)
    Njw::TI       # working years (retire at Njw+1: enter at 21 and retire at 65)
    Na::TI       # asset state
    Na2::TI     # asset choice
    Ne::TI         # labor productivity (low/high)

    # minA::TF       # min asset grid
    maxA::TF      # max asset grid
    curvA::TF    # asset grid density (=1 eequal size)

    meaJ::Array{TF,1}
    L::TF
    gride::Array{TF,1}
    Pe::Array{TF,2}

    b::TF                 # borrowing limit

    NT::TI # transition period
end


In [5]:
m = Model(
    alpha, delta, beta, rho,
    maxiter, tol, adjK,
    Nj, Njw, Na, Na2, Ne,
    # minA, 
    maxA, curvA,
    meaJ, L, gride, Pe,
    0.0, # no borrowing
    NT,
);

In [6]:
function generate_capital_grid(m, r=nothing, wage=nothing)

    if (r === nothing) && (wage === nothing) # if these are not specified, use no-borrowing condition
        phi = 0.0;
    else
        # borrowing limit
        if r <= 0.0
            phi = m.b;
        else
            phi = min(m.b,wage*m.s[1]/r);
        end
    end

    # -phi is borrowing limit, b is adhoc
    # the second term is natural limit

    # capital grid (need define in each iteration since it depends on r/phi)
    minA = -phi;                                  # borrowing constraint

    # asset state grid
    grida = zeros(Na);
    grida[1] = minA;
    for ac in 2:Na
        grida[ac] = grida[1] + (maxA-minA)*((ac-1)/(Na-1))^curvA;
    end

    # asset choice grid
    grida2 = zeros(Na2);
    grida2[1] = minA;
    for ac in 2:Na2
        grida2[ac] = grida2[1] + (maxA-minA)*((ac-1)/(Na2-1))^curvA;
    end

    return grida, grida2
end
grids = generate_capital_grid(m);
grida, grida2 = grids;

In [7]:
function translate_capital_grid(m, grids)

    grida, grida2 = grids

    # =================================================== #
    #  SPLIT GRID in grida2 TO NEARBY TWO GRIDS IN grida  #
    # =================================================== #
    
    # calculate node and weight for interpolation  
    ac1vec=zeros(Int64, m.Na2);
    ac2vec=zeros(Int64, m.Na2);

    pra1vec=zeros(m.Na2);
    pra2vec=zeros(m.Na2);

    for ac in 1:m.Na2

        xx = grida2[ac];

        if xx >= grida[m.Na]

            ac1vec[ac] = m.Na;
            ac2vec[ac] = m.Na;

            pra1vec[ac] = 1.0;
            pra2vec[ac] = 0.0;

        else

            ind = 1;
            while xx > grida[ind+1]
                ind += 1
                if ind+1 >= m.Na
                    break
                end
            end

            ac1vec[ac] = ind

            if ind < m.Na

                ac2vec[ac] = ind+1;
                dA=(xx-grida[ind])/(grida[ind+1]-grida[ind]);
                pra1vec[ac] = 1.0-dA;
                pra2vec[ac] = dA;

            else

                ac2vec[ac] = ind;
                pra1vec[ac] = 1.0;
                pra2vec[ac] = 0.0;

            end
        end
    end

    return ac1vec, ac2vec, pra1vec, pra2vec
end
capital_grid_translations = translate_capital_grid(m, grids);
ac1vec, ac2vec, pra1vec, pra2vec = capital_grid_translations;

In [8]:
function solve_value_function(m, grids, capital_grid_translations)

    # asset grids
    grida, grida2 = grids;

    # mapping between asset variables
    ac1vec, ac2vec, pra1vec, pra2vec = capital_grid_translations;

    # value function/solution (initialization)
    vfun = zeros(m.Nj, m.Ne, m.Na);
    afunG = zeros(Int64, m.Nj, m.Ne, m.Na);
    afun = zeros(m.Nj, m.Ne, m.Na);

    # income grid (initialization)
    yvec = zeros(m.Nj,m.Ne);

    # distribution (initialization)
    mea = zeros(m.Nj,m.Ne,m.Na)

    # equilibrium tax rate
    tau = m.rho*sum(m.meaJ[m.Njw+1:m.Nj])/sum(m.meaJ[1:m.Njw]);

    # initial guess of K
    K = 7.0;
    r = 0.0;

    # wage (initialization)
    w = 0.0;

    for iter = 1:m.maxiter

        # compute r/w/SS
        r = m.alpha*(K/L)^(m.alpha-1) - m.delta;
        w = (1-m.alpha)*(K/L)^m.alpha;
        ss = m.rho * w
        
        # net income by age
        for jc in 1:m.Njw
            for ec in 1:m.Ne
                yvec[jc,ec] = (1-tau) * w * m.gride[ec]
            end
        end
        yvec[m.Njw+1:m.Nj,:] .= ss;


        # household problem: from last age Nj to 1 (backwards)
        
        # (1) age Nj (last age)
        jc = m.Nj;
        for ec in 1:m.Ne
            for ac in 1:m.Na
                
                c = yvec[jc,ec] + (1+r) * grida[ac];
                vfun[jc,ec,ac] = log(c);
                afunG[jc,ec,ac] = 1; # no saving
                afun[jc,ec,ac] = grida2[1];

            end
        end

        # (2) age Nj-1:1
        for jc in m.Nj-1:-1:1
            for ec in 1:m.Ne

                y = yvec[jc,ec];

                for ac in 1:m.Na

                    vtemp = -1000000 .* ones(m.Na2); # initialization (store values)
                    accmax = m.Na2;

                    for acc in 1:m.Na2

                        c = y + (1+r) * grida[ac] - grida2[acc];

                        if c <= 0.0
                            accmax = acc-1;
                            break
                        end

                        acc1 = ac1vec[acc];
                        acc2 = ac2vec[acc];

                        vpr = 0.0;
                        for ecc in 1:m.Ne
                            vpr += m.Pe[ec,ecc] * (
                                pra1vec[acc] * vfun[jc+1,ecc,acc1] 
                                + pra2vec[acc] * vfun[jc+1,ecc,acc2])
                                ;
                        end

                        vtemp[acc] = log(c) + m.beta * vpr;

                    end

                    val,index = findmax(vtemp[1:accmax]);
                    vfun[jc,ec,ac] = val;
                    afunG[jc,ec,ac] = index; # grid from grida2
                    afun[jc,ec,ac] = grida2[index];

                end
            end
        end

        # compute distribution mea
        mea = zeros(m.Nj,m.Ne,m.Na);    # initialization
        mea[1,:,1] .= m.meaJ[1]/m.Ne; # zero asset at age 1

        for jc in 1:m.Nj-1
            for ec in 1:m.Ne
                for ac in 1:m.Na

                    mea0 = mea[jc,ec,ac];
                    acc = afunG[jc,ec,ac];
                    
                    acc1 = ac1vec[acc];
                    acc2 = ac2vec[acc];

                    pra1 = pra1vec[acc];
                    pra2 = pra2vec[acc];

                    for ecc in 1:m.Ne
                        
                        mea[jc+1,ecc,acc1] += m.Pe[ec,ecc]*pra1*mea0;
                        mea[jc+1,ecc,acc2] += m.Pe[ec,ecc]*pra2*mea0;

                    end
                end
            end
        end

        errm = abs(sum(mea)-1);
        if errm > 1e-4
            println("error in computation of distribution: $errm")
            break
        end

        mea_maxA = sum(mea[:,:,m.Na]);
        if mea_maxA > 1e-3
            println("measure at max asset grid is large: $mea_maxA")
        end

        # compute error in K=A
        A = sum(afun.*mea);
        errK = abs(K-A);

        if errK < m.tol
            println("K coverged: $iter")
            break
        end

        if iter > m.maxiter
            println("WARN: iter>maxiter: $iter, $errK")
        end

        # update Kitao
        K += m.adjK*(A-K);

        println("$iter errK = $errK")

    end

    return (; vfun, afun, afunG, mea, r, w, K)

end


solve_value_function (generic function with 1 method)

In [9]:
res = solve_value_function(m, grids, capital_grid_translations);

1 errK = 2.9561332195148493
2 errK = 0.9493548131707792
3 errK = 0.2203752176811795
4 errK = 0.045122494864247464
5 errK = 0.00835506937669983
6 errK = 0.0019005269091634247
K coverged: 7


In [16]:
# =============== #
#  COMPUTE STATS  #
# =============== #

# asset by age
afunJ = zeros(m.Nj);
for jc in 1:m.Nj
    tempA = 0.0;
    for ac in 1:m.Na
        tempA += grids[1][ac] * sum(res.mea[jc,:,ac])
    end
    afunJ[jc] = tempA / m.meaJ[jc]
end 

# asset by age/skill
afunJE = zeros(m.Nj,m.Ne);
for jc in 1:m.Nj
    for ec in 1:m.Ne
        temp = 0.0;
        for ac in 1:m.Na
            temp += grids[1][ac] * res.mea[jc,ec,ac];
        end
        afunJE[jc,ec] = temp/sum(res.mea[jc,ec,:]);
    end
end

cfun = zeros(m.Nj,m.Ne,m.Na); # consumption function
sfun = zeros(m.Nj,m.Ne,m.Na); # saving function
srat = zeros(m.Nj,m.Ne,m.Na); # stores saving rate given state

tau = m.rho*sum(m.meaJ[m.Njw+1:m.Nj])/sum(m.meaJ[1:m.Njw]);
ss = m.rho * res.w
# net income by age
yvec = zeros(m.Nj,m.Ne);
for jc in 1:m.Njw
    for ec in 1:m.Ne
        yvec[jc,ec] = (1-tau) * res.w * m.gride[ec]
    end
end
yvec[m.Njw+1:m.Nj,:] .= ss;

for jc in 1:m.Nj
    for ec in 1:m.Ne
        for ac in 1:m.Na

            acc = res.afunG[jc,ec,ac];
            y = yvec[jc,ec];
            c = y + (1+res.r)*grids[1][ac] - grids[2][acc];
            inc = y + res.r*grids[1][ac];

            sfun[jc,ec,ac] = inc - c;
            cfun[jc,ec,ac] = c;
            srat[jc,ec,ac] = (inc-c)/inc;

        end
    end
end

# consumption by age
cfunJ = zeros(m.Nj);
for jc in 1:m.Nj
    cfunJ[jc] = sum(res.mea[jc,:,:] .* cfun[jc,:,:]) / sum(res.mea[jc,:,:]);
end

# saving and saving rate by age/skill
sfunJE = zeros(m.Nj,m.Ne);
sratJE = zeros(m.Nj,m.Ne);
for jc in 1:m.Nj
    for ec in 1:m.Ne

        sfunJE[jc,ec] = sum(res.mea[jc,ec,:] .* sfun[jc,ec,:]) / sum(res.mea[jc,ec,:]);
        sratJE[jc,ec] = sum(res.mea[jc,ec,:] .* srat[jc,ec,:]) / sum(res.mea[jc,ec,:]);

    end
end

age = collect(1:m.Nj);
age .+= 19;

norm = 1/cfunJ[1];

minJ = 20;
maxJ = 80;


plot(age,norm.*afunJE[:,2],ls=:solid,lc=:black,lw=3,label="high",legend=:topleft)
plot!(age,norm.*afunJE[:,1],ls=:dashdot,lc=:black,lw=3,label="low")
xlims!(minJ,maxJ)
title!("Asset (Stock) By Age and Prod")
xlabel!("Age")
ylabel!("Asset")
savefig("images/fig_olg2_a.pdf")


# saving rate by asset: choose a particular age to plot
jc = 21;
sfunAE = zeros(Na,Ne);
sratAE = zeros(Na,Ne);
for ac in 1:Na
    for ec in 1:Ne

        sfunAE[ac,ec] = sfun[jc,ec,ac];
        sratAE[ac,ec] = srat[jc,ec,ac];

    end
end

minA = 0.0;
maxA = 20.0;
minY = -0.2;
maxY = 0.5;
zerovec = zeros(Na);

plot(grids[1],sratAE[:,2],ls=:solid,lc=:black,lw=3,label="high")
plot!(grids[1],sratAE[:,1],ls=:dashdot,lc=:black,lw=3,label="low")
plot!(grids[1],zerovec,ls=:dot,lc=:black,lw=1,label="")
xlims!(minA,maxA)
title!("Saving Rate By Asset and Prod")
xlabel!("Asset")
ylabel!("Saving Rate")
savefig("images/fig_olg2_s.pdf")


plot(grids[1],sfunAE[:,2],ls=:solid,lc=:black,lw=3,label="high")
plot!(grids[1],sfunAE[:,1],ls=:dashdot,lc=:black,lw=3,label="low")
plot!(grids[1],zerovec,ls=:dot,lc=:black,lw=1,label="")
xlims!(minA,maxA)
title!("Saving Level By Asset and Prod")
xlabel!("Asset")
ylabel!("Saving (level)")
savefig("images/fig_olg2_slevel.pdf")


"/Users/mizuhirosuzuki/Documents/GitHub/quant_macro_study_group/Ch6/images/fig_olg2_slevel.pdf"

## 移行過程を解く

In [17]:
rho0 = 0.50; # replacement rate at initial steady state
rho1 = 0.25; # replacement rate at final steady state
m_ss0 = Model(
    alpha, delta, beta, rho0,
    maxiter, tol, adjK,
    Nj, Njw, Na, Na2, Ne,
    # minA, 
    maxA, curvA,
    meaJ, L, gride, Pe,
    0.0, # no borrowing
    NT
);
m_ss1 = Model(
    alpha, delta, beta, rho1,
    maxiter, tol, adjK,
    Nj, Njw, Na, Na2, Ne,
    # minA, 
    maxA, curvA,
    meaJ, L, gride, Pe,
    0.0, # no borrowing
    NT
);

In [18]:
res_ss0 = solve_value_function(m_ss0, grids, capital_grid_translations);
res_ss1 = solve_value_function(m_ss1, grids, capital_grid_translations);

1 errK = 2.9561332195148493
2 errK = 0.9493548131707792
3 errK = 0.2203752176811795
4 errK = 0.045122494864247464
5 errK = 0.00835506937669983
6 errK = 0.0019005269091634247
K coverged: 7
1 errK = 0.5218646107989091
2 errK = 0.18864564790422111
3 errK = 0.06623808490654781
4 errK = 0.02265598722412232
5 errK = 0.008124898677743353
6 errK = 0.0033729001677613724
K coverged: 7


In [19]:
K_SS0 = res_ss0.K;
mea_SS0 = res_ss0.mea;
vfun_SS0 = res_ss0.vfun;
K_SS1 = res_ss1.K;
mea_SS1 = res_ss1.mea;
vfun_SS1 = res_ss1.vfun;

### 移行過程の計算

In [20]:
# ======================= #
#  TRANSITION SUB-ROUTINES  #
# ======================= #

# Path of replacement rate and balanced-budget tax rate.
function build_policy_paths(m, rho_init, rho_final; TT=25)
    rhoT = zeros(m.NT)
    for tc in 1:TT
        rhoT[tc] = rho_init + ((rho_final - rho_init)/(TT-1))*(tc-1)
    end
    rhoT[TT+1:m.NT] .= rho_final

    retired = sum(m.meaJ[m.Njw+1:m.Nj])
    working = sum(m.meaJ[1:m.Njw])
    tauT = zeros(m.NT)
    for tc in 1:m.NT
        tauT[tc] = rhoT[tc] * retired / working
    end
    return rhoT, tauT
end

# Linear initial guess for the capital path: K_SS0 -> K_SS1 over NT0 periods.
function initial_guess_KT(m, K_SS0, K_SS1; NT0=30)
    KT0 = K_SS1 .* ones(m.NT)
    intK = (K_SS1 - K_SS0)/(NT0 - 1)
    for tc in 1:NT0
        KT0[tc] = K_SS0 + intK*(tc-1)
    end
    return KT0
end

# Net income by age/productivity given prices and policy.
function income_grid(m, w, tau, rho)
    yvec = zeros(m.Nj, m.Ne)
    for jc in 1:m.Njw, ec in 1:m.Ne
        yvec[jc,ec] = (1-tau)*w*m.gride[ec]
    end
    yvec[m.Njw+1:m.Nj,:] .= rho*w
    return yvec
end

# Solve the household problem at a single date given continuation value vfun_next.
# Speed tricks:
#   monotonicity — optimal acc*(ac) is non-decreasing in ac, so scan starts
#   from the previous ac's optimum. concavity — once value stops rising, break.
function solve_household_period(m, grids, capital_grid_translations, vfun_next, r, yvec)
    grida, grida2 = grids
    ac1vec, ac2vec, pra1vec, pra2vec = capital_grid_translations

    vfun1 = zeros(m.Nj, m.Ne, m.Na)
    afunG = zeros(Int64, m.Nj, m.Ne, m.Na)
    afun  = zeros(m.Nj, m.Ne, m.Na)

    # last age: no saving, consume everything
    jc = m.Nj
    @inbounds for ec in 1:m.Ne, ac in 1:m.Na
        c = yvec[jc,ec] + (1+r)*grida[ac]
        vfun1[jc,ec,ac] = log(c)
        afunG[jc,ec,ac] = 1
        afun[jc,ec,ac]  = grida2[1]
    end

    # ages Nj-1 down to 1
    for jc in m.Nj-1:-1:1
        for ec in 1:m.Ne
            y = yvec[jc,ec]
            acc_lo = 1  # monotone lower bound, grows with ac
            @inbounds for ac in 1:m.Na
                cash = y + (1+r)*grida[ac]
                best_val = -Inf
                best_idx = acc_lo
                for acc in acc_lo:m.Na2
                    c = cash - grida2[acc]
                    if c <= 0.0
                        break  # infeasible, and all larger acc too
                    end
                    acc1 = ac1vec[acc]; acc2 = ac2vec[acc]
                    vpr = 0.0
                    for ecc in 1:m.Ne
                        vpr += m.Pe[ec,ecc] * (
                            pra1vec[acc]*vfun_next[jc+1,ecc,acc1]
                          + pra2vec[acc]*vfun_next[jc+1,ecc,acc2])
                    end
                    v = log(c) + m.beta*vpr
                    if v > best_val
                        best_val = v
                        best_idx = acc
                    else
                        break  # concavity: past the peak
                    end
                end
                vfun1[jc,ec,ac] = best_val
                afunG[jc,ec,ac] = best_idx
                afun[jc,ec,ac]  = grida2[best_idx]
                acc_lo = best_idx
            end
        end
    end
    return vfun1, afunG, afun
end

# Backward sweep over the transition path, tc = NT -> 1.
function backward_vfi_transition(m, grids, capital_grid_translations,
                                  KT0, rhoT, tauT, vfun_SS1; iterTR=0)
    afunGT   = zeros(Int64, m.NT, m.Nj, m.Ne, m.Na)
    afunT    = zeros(m.NT, m.Nj, m.Ne, m.Na)
    vfun_TR  = zeros(m.NT, m.Ne)
    vfun_TR0 = zeros(m.Nj, m.Ne, m.Na)

    vfun_next = copy(vfun_SS1)
    for tc in m.NT:-1:1
        r = m.alpha * (KT0[tc]/m.L)^(m.alpha-1) - m.delta
        w = (1-m.alpha) * (KT0[tc]/m.L)^m.alpha
        yvec = income_grid(m, w, tauT[tc], rhoT[tc])

        vfun1, afunG, afun = solve_household_period(
            m, grids, capital_grid_translations, vfun_next, r, yvec)

        afunGT[tc,:,:,:] .= afunG
        afunT[tc,:,:,:]  .= afun
        vfun_TR[tc,:]     = vfun1[1,:,1]
        if tc == 1
            vfun_TR0 = copy(vfun1)
        end
        vfun_next = vfun1
    end
    return afunGT, afunT, vfun_TR, vfun_TR0
end

# Forward simulation of the distribution, tc = 1 -> NT.
function forward_distribution_transition(m, capital_grid_translations, afunGT, mea_SS0)
    ac1vec, ac2vec, pra1vec, pra2vec = capital_grid_translations

    meaT = zeros(m.NT, m.Nj, m.Ne, m.Na)
    meaT[1,:,:,:] .= mea_SS0
    mea0 = copy(mea_SS0)
    mea1 = zeros(m.Nj, m.Ne, m.Na)

    errm = sum(mea0) - 1
    if errm > 1e-4
        error("initial distribution does not sum to 1: errm = $errm")
    end

    for tc in 1:m.NT-1
        fill!(mea1, 0.0)
        mea1[1,:,1] .= m.meaJ[1]/m.Ne

        @inbounds for jc in 1:m.Nj-1, ec in 1:m.Ne, ac in 1:m.Na
            meaX = mea0[jc,ec,ac]
            meaX == 0.0 && continue
            acc  = afunGT[tc,jc,ec,ac]
            acc1 = ac1vec[acc]; acc2 = ac2vec[acc]
            pra1 = pra1vec[acc]; pra2 = pra2vec[acc]
            for ecc in 1:m.Ne
                p = m.Pe[ec,ecc]*meaX
                mea1[jc+1,ecc,acc1] += p*pra1
                mea1[jc+1,ecc,acc2] += p*pra2
            end
        end

        errm = abs(sum(mea1) - 1)
        if errm > 1e-4
            error("distribution does not sum to 1: errm = $errm, tc = $tc")
        end
        mea_maxA = sum(@view mea1[:,:,m.Na])
        if mea_maxA > 1e-3
            println("measure at max asset grid is large: mea_maxA = $mea_maxA, tc = $tc")
            flush(stdout)
        end

        meaT[tc+1,:,:,:] .= mea1
        mea0, mea1 = mea1, mea0  # swap buffers to avoid realloc
    end
    return meaT
end

# Aggregate capital path from distribution and savings policy.
function aggregate_capital_path(m, afunT, meaT, KT0)
    KT1   = zeros(m.NT)
    errKT = zeros(m.NT)
    KT1[1] = KT0[1]  # predetermined
    @inbounds for tc in 1:m.NT-1
        s = 0.0
        for jc in 1:m.Nj, ec in 1:m.Ne, ac in 1:m.Na
            s += meaT[tc,jc,ec,ac] * afunT[tc,jc,ec,ac]
        end
        KT1[tc+1]   = s
        errKT[tc+1] = abs(KT1[tc] - KT0[tc])
    end
    return KT1, errKT
end


aggregate_capital_path (generic function with 1 method)

In [21]:
function compute_transition(
    m, grids, capital_grid_translations,
    K_SS0, K_SS1, rho0, rho1,
    vfun_SS1=vfun_SS1, mea_SS0=mea_SS0;
    maxiterTR=300, errKTol=1e-4, TT=25, NT0=30,
)
    rhoT, tauT = build_policy_paths(m, rho0, rho1; TT=TT)
    KT0 = initial_guess_KT(m, K_SS0, K_SS1; NT0=NT0)

    vfun_TR  = zeros(m.NT, m.Ne)
    vfun_TR0 = zeros(m.Nj, m.Ne, m.Na)
    errKvec  = zeros(maxiterTR)
    errK     = 1.0
    iterTR   = 1

    while (errK > errKTol) && (iterTR < maxiterTR)

        afunGT, afunT, vfun_TR, vfun_TR0 = backward_vfi_transition(
            m, grids, capital_grid_translations,
            KT0, rhoT, tauT, vfun_SS1; iterTR=iterTR)

        meaT = forward_distribution_transition(
            m, capital_grid_translations, afunGT, mea_SS0)

        KT1, errKT = aggregate_capital_path(m, afunT, meaT, KT0)

        errK = maximum(errKT)
        errKvec[iterTR] = errK

        if errK > errKTol
            for tc in 2:m.NT
                KT0[tc] += m.adjK*(KT1[tc] - KT0[tc])
            end
        else
            println("converged: iterTR = $iterTR, errK = $errK"); flush(stdout)
        end

        println("iterTR = $iterTR, errK = $errK"); flush(stdout)
        iterTR += 1
    end

    if iterTR == maxiterTR
        println("iteration not converged: iterTR = $iterTR, errK = $errK"); flush(stdout)
    end

    return (; KT0, vfun_TR, vfun_TR0)
end


compute_transition (generic function with 3 methods)

In [22]:
res_transition = compute_transition(
    m, grids, capital_grid_translations, K_SS0, K_SS1, rho0, rho1
    )

iterTR = 1, errK = 0.2689805891726609
iterTR = 2, errK = 0.14674162465939133
iterTR = 3, errK = 0.08487855330558602
iterTR = 4, errK = 0.05222907511420516
iterTR = 5, errK = 0.033661917766727
iterTR = 6, errK = 0.023007309773180218
iterTR = 7, errK = 0.016202405379370788
iterTR = 8, errK = 0.01165061918190613
iterTR = 9, errK = 0.008561399644124279
iterTR = 10, errK = 0.006348302462012789
iterTR = 11, errK = 0.00475988445723452
iterTR = 12, errK = 0.0036153015285567136
iterTR = 13, errK = 0.0027351193399658413
iterTR = 14, errK = 0.002082734956470489
iterTR = 15, errK = 0.0016021955593608084
iterTR = 16, errK = 0.0012401845548524193
iterTR = 17, errK = 0.0009486755430225813
iterTR = 18, errK = 0.0007401410328009561
iterTR = 19, errK = 0.0005799838911180188
iterTR = 20, errK = 0.0004513404967303458
iterTR = 21, errK = 0.0003490697988768332
iterTR = 22, errK = 0.0002710245998649441
iterTR = 23, errK = 0.00021420421004059165
iterTR = 24, errK = 0.00016383571902078842
iterTR = 25, errK = 0

(KT0 = [6.163751731696616, 6.1874852021088484, 6.211197928884058, 6.234651872208942, 6.257837554903042, 6.280763683835439, 6.303484210813301, 6.32595372154781, 6.348207400858465, 6.370226091019567  …  6.837104319591722, 6.837131637048467, 6.837162587866509, 6.837197255583895, 6.837231583754566, 6.8372651125884065, 6.837295356134617, 6.837324256586303, 6.837352029578276, 6.83737475273528], vfun_TR = [3.6350231994383 5.442635890542027; 3.7524308155154955 5.558871448391541; … ; 5.381141455607435 7.214011589217731; 5.381171318421799 7.214047807613749], vfun_TR0 = [3.6350231994383 5.442635890542027; 3.4482840903634444 5.258766533587669; … ; -0.7216575646152531 -0.7216575646152531; -0.35481399876285635 -0.35481399876285635;;; 3.688930984924988 5.48918544037538; 3.5021938806496653 5.3054081470207235; … ; -0.6579872739783801 -0.6579872739783801; -0.2930189952447212 -0.2930189952447212;;; 3.7570822930460936 5.549465267179788; 3.5703641438474 5.365801479440049; … ; -0.5786232743898403 -0.5786232

In [23]:

# plot transition dynamics
rT = zeros(m.NT);
wT = zeros(m.NT);

for tc in 1:m.NT

    rT[tc] = m.alpha * ((res_transition.KT0[tc]/m.L)^(m.alpha-1)) - m.delta;
    wT[tc] = (1-m.alpha) * (res_transition.KT0[tc]/m.L)^m.alpha;

end

maxY = 100;
norm = copy(K_SS0);
gridT = collect(1:m.NT);

plot([1],[K_SS0/norm], mc=:red, markershapes=:circle, lw=2, legend=false)
plot!(gridT, res_transition.KT0./norm, color=:blue, lw=2)
plot!([maxY], [K_SS1/norm], mc=:red, markershapes=:circle, lw=2)
title!("Capital")
xlabel!("Time")
xlims!(1-0.9,maxY+0.9)
savefig("images/fig_olg2_tr_k.pdf")


plot([1],[rT[1]], mc=:red, markershapes=:circle, lw=2, legend=false)
plot!(gridT, rT, color=:blue, lw=2)
plot!([maxY], [rT[NT]], mc=:red, markershapes=:circle, lw=2)
title!("Interest Rate")
xlabel!("Time")
xlims!(1-0.9,maxY+0.9)
savefig("images/fig_olg2_tr_r.pdf")

"/Users/mizuhirosuzuki/Documents/GitHub/quant_macro_study_group/Ch6/images/fig_olg2_tr_r.pdf"

In [27]:
betaJ = zeros(m.Nj);

for jc in 1:m.Nj
    temp = 0.0;
    for ic in jc:m.Nj
        temp += m.beta^(ic-jc);
    end
    betaJ[jc] = temp;
end 

welf0 = zeros(m.Nj,m.Ne,m.Na);
for jc in 1:m.Nj
    for ec in 1:m.Ne
        for ac in 1:m.Na
            welf0[jc,ec,ac] = exp(
                (res_transition.vfun_TR0[jc,ec,ac] - vfun_SS0[jc,ec,ac]) / betaJ[jc] 
            ) - 1;
        end
    end
end

welf0_JE = zeros(m.Nj,m.Ne);
for jc in 1:m.Nj
    for ec in 1:m.Ne
        welf0_JE[jc,ec] = sum(
            welf0[jc,ec,:].*mea_SS0[jc,ec,:]
        ) / sum(mea_SS0[jc,ec,:]);
    end
end

welfTR = zeros(m.NT,m.Ne);
for tc in 1:m.NT
    for ec in 1:m.Ne
        welfTR[tc,ec] = exp(
            (res_transition.vfun_TR[tc,ec] - vfun_SS0[1,ec,1]) / betaJ[1] 
        ) - 1;
    end
end 

time = collect(1:m.NT);

plot(time, welfTR[:,1], ls=:dash, c=:black, lw=3, label="low", legend=:bottomright)
plot!(time, welfTR[:,2], ls=:solid, c=:black, lw=3, label="high")
xlims!(1,60)
ylims!(0.01,0.08)
xlabel!("Cohort")
ylabel!("CEV")
savefig("images/fig_welf_tr.pdf")


age = collect(1:m.Nj);
age .+= 19;

plot(age, welf0_JE[:,1], ls=:dash, c=:black, lw=3, label="low", legend=:bottomright)
plot!(age, welf0_JE[:,2], ls=:solid, c=:black, lw=3, label="high")
ylims!(-0.1,0.04)
xlabel!("Age")
ylabel!("CEV")
savefig("images/fig_welf_0.pdf")


jc = 21; # 40 yrs old
welf0_AE = zeros(m.Na,m.Ne);
for ac in 1:m.Na
    for ec in 1:m.Ne
        welf0_AE[ac,ec] = welf0[jc,ec,ac];
    end
end

plot(grida, welf0_AE[:,1], ls=:dash, c=:black, lw=3, label="low", legend=:bottomright)
plot!(grida, welf0_AE[:,2], ls=:solid, c=:black, lw=3, label="high")
xlabel!("Asset")
ylabel!("CEV")
savefig("images/fig_welf_aged_40.pdf")


"/Users/mizuhirosuzuki/Documents/GitHub/quant_macro_study_group/Ch6/images/fig_welf_aged_40.pdf"